# C0 · Move the pre-G1 work from Kaggle to Colab

Run this **once**. It puts everything G1 and G2 need into your Google Drive, under `MyDrive/ReNeg/`:

| What | From | To |
|---|---|---|
| oodlab code | your private Kaggle dataset `oodlab-code` (or `oodlab_code.zip` you upload) | `code/oodlab_code/` |
| reneg code, notebooks C1 and G1 | `ReNeg_Colab_bundle.zip` you upload | `code/reneg_code/`, `notebooks/` |
| ViT-B/16 feature cache (12 sets + text bank, about 0.3 GB) | output of Kaggle notebook 01 | `oodlab_working/cache/vit-b-16/` |
| results of Kaggle notebooks 02–04 | their outputs | `oodlab_working/results/` |
| Colab copies of Kaggle notebooks 00–04 | the notebooks themselves | `notebooks_colab/` |

**Before running** (GUIDE_COLAB.md steps 1–3): upload `ReNeg_Colab_bundle.zip` into `MyDrive/ReNeg/uploads/`
and add your Kaggle key to Colab Secrets. Without a Kaggle key the notebook uses whatever zips you put in
`uploads/` instead (guide route C).

**Runtime:** CPU is enough (*Runtime → Change runtime type → CPU*). About 5–10 minutes. Run the cells in order
with *Runtime → Run all*; every cell can be run again safely.

In [ ]:
# ── Step 1 · Drive folders and the ReNeg code ──────────────────────────────────────────
import os, sys, shutil, zipfile
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
for sub in ("uploads", "code", "notebooks", "transfers"):
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)

def find_bundle_root(folder, depth=4):
    """Folder that contains reneg_code/reneg/__init__.py (searched a few levels deep)."""
    for dp, dn, fn in os.walk(folder):
        if dp[len(folder):].count(os.sep) > depth:
            dn[:] = []
            continue
        if os.path.isfile(os.path.join(dp, "reneg_code", "reneg", "__init__.py")):
            return dp
    return None

uploads = os.path.join(DRIVE_ROOT, "uploads")
bundle_root = None
for f in sorted(os.listdir(uploads), reverse=True):     # newest bundle name last-sorted wins
    p = os.path.join(uploads, f)
    if f.lower().endswith(".zip"):
        with zipfile.ZipFile(p) as z:
            if any(n.endswith("reneg_code/reneg/__init__.py") for n in z.namelist()):
                dest = os.path.join(DRIVE_ROOT, "transfers", "bundle")
                shutil.rmtree(dest, ignore_errors=True)
                z.extractall(dest)
                bundle_root = find_bundle_root(dest)
                print("unpacked", f)
                break
    elif os.path.isdir(p) and find_bundle_root(p):
        bundle_root = find_bundle_root(p)
        break
RENEG_CODE = os.path.join(DRIVE_ROOT, "code", "reneg_code")
if bundle_root:
    shutil.rmtree(RENEG_CODE, ignore_errors=True)
    shutil.copytree(os.path.join(bundle_root, "reneg_code"), RENEG_CODE)
    for f in os.listdir(os.path.join(bundle_root, "notebooks")):
        if f.endswith(".ipynb") and not f.startswith("C0"):
            shutil.copy2(os.path.join(bundle_root, "notebooks", f), os.path.join(DRIVE_ROOT, "notebooks", f))
    if os.path.isfile(os.path.join(bundle_root, "GUIDE_COLAB.md")):
        shutil.copy2(os.path.join(bundle_root, "GUIDE_COLAB.md"), os.path.join(DRIVE_ROOT, "GUIDE_COLAB.md"))
if not os.path.isfile(os.path.join(RENEG_CODE, "reneg", "__init__.py")):
    raise FileNotFoundError("ReNeg_Colab_bundle.zip not found in MyDrive/ReNeg/uploads/ (GUIDE_COLAB.md step 2)")
if RENEG_CODE not in sys.path:
    sys.path.insert(0, RENEG_CODE)
import importlib
import reneg
from reneg import colab as rc
reneg, rc = importlib.reload(reneg), importlib.reload(rc)   # pick up a newer bundle without restarting
L = rc.layout(DRIVE_ROOT)
print(f"reneg {reneg.__version__} installed in {RENEG_CODE}; notebooks copied to {L['notebooks']}")

## Step 2 · Kaggle access

Reads `KAGGLE_USERNAME` and `KAGGLE_KEY` from Colab Secrets (the key icon on the left; switch on *Notebook
access*). If they are missing, the notebook keeps going with the zips in `uploads/`.

In [ ]:
KAGGLE_USER = rc.setup_kaggle_credentials()
USE_KAGGLE_API = bool(KAGGLE_USER)
if USE_KAGGLE_API:
    rc.ensure_kaggle_cli()
print("Kaggle API:", f"on (user {KAGGLE_USER})" if USE_KAGGLE_API else "off -> using the files in MyDrive/ReNeg/uploads")

## Step 3 · Your Kaggle names

A notebook's slug is the last part of its URL: `kaggle.com/code/<user>/<slug>`. The list printed below shows
notebooks on your account. Each of the five is marked **found** (matched in the list) or **guessed** (not in
the list, so the slug is made the way Kaggle makes it: `Notebook_three` → `notebook-three`). A wrong guess
shows up as a FAILED download in step 5; then copy the real slug from the notebook's URL into `NOTEBOOKS`
and run from this cell again.

In [ ]:
CODE_DATASET = "oodlab-code"                # your private Kaggle dataset holding oodlab_code.zip
NOTEBOOKS = {                               # your Kaggle notebooks (title or slug; case, _ and - don't matter)
    "00": "notebook0",
    "01": "notebook1",                      # builds the feature cache: the one that matters most
    "02": "notebook2",
    "03": "Notebook_three",
    "04": "Notebook_four",
}
EXTRA_DATASETS = []                         # guide route B, e.g. ["your-user/reneg-01-output"]
if USE_KAGGLE_API:
    rows = rc.list_my_kernels()
    print(f"{len(rows)} notebooks listed on your Kaggle account:")
    for r in rows[:40]:
        print("   ", r.get("ref"), "|", r.get("title"), "|", r.get("lastRunTime"))
    print()
    NOTEBOOKS = rc.resolve_or_guess(NOTEBOOKS, rows)      # real (or guessed) slugs from here on

## Step 4 · oodlab code

In [ ]:
kaggle_dir = os.path.join(L["transfers"], "kaggle")
if USE_KAGGLE_API:
    rc.download_dataset(f"{KAGGLE_USER}/{CODE_DATASET}", os.path.join(kaggle_dir, CODE_DATASET))
rc.unpack_uploads(L)                        # zips you uploaded by hand (oodlab_code.zip, notebook outputs)
rc.expand_nested_zips(L["transfers"])       # a Kaggle dataset may contain oodlab_code.zip itself
got = rc.install_code([os.path.join(kaggle_dir, CODE_DATASET), os.path.join(L["transfers"], "uploads")],
                      "oodlab", L["oodlab_code"])
print("oodlab code:", got or "NOT FOUND -> upload oodlab_code.zip to MyDrive/ReNeg/uploads and run this cell again")

## Step 5 · Feature cache and results

Downloads the outputs of Kaggle notebooks 01–04 (their `/kaggle/working` folders) and installs the cache and
the results. Notebook 01's output is the big one (about 0.3 GB).

In [ ]:
if USE_KAGGLE_API:
    for k, slug in NOTEBOOKS.items():
        if k == "00":
            continue
        ok = rc.download_kernel_output(f"{KAGGLE_USER}/{slug}", os.path.join(kaggle_dir, slug))
        print(f"{slug}: {'downloaded' if ok else 'FAILED -> GUIDE_COLAB.md route B or C'}")
for slug in EXTRA_DATASETS:                 # route B: a Kaggle dataset made from a notebook's output
    ok = rc.download_dataset(slug, os.path.join(kaggle_dir, "datasets", slug.split("/")[-1]))
    print(f"{slug}: {'downloaded' if ok else 'FAILED'}")
rc.unpack_uploads(L)
search = [kaggle_dir, os.path.join(L["transfers"], "uploads")]
tags = rc.install_cache(search, L["cache"])
names = rc.install_results(search, L["results"])
print("caches installed:", tags or "nothing new")
print("results installed:", names or "nothing new")

## Step 6 · Colab copies of your Kaggle notebooks (optional)

Each copy gets the Colab set-up cell in place of the Kaggle one, so it runs from Drive. Notebooks you upload
by hand (`*.ipynb` in `uploads/`) are converted too.

In [ ]:
converted = []
if USE_KAGGLE_API:
    for k, slug in NOTEBOOKS.items():
        src = rc.pull_kernel(f"{KAGGLE_USER}/{slug}", os.path.join(L["transfers"], "kaggle_notebooks", slug))
        if src:
            dst = os.path.join(L["notebooks_colab"], f"{slug}_colab.ipynb")
            converted.append((dst, rc.convert_notebook(src, dst)))
for f in sorted(os.listdir(L["uploads"])):
    if f.endswith(".ipynb"):
        dst = os.path.join(L["notebooks_colab"], f.replace(".ipynb", "_colab.ipynb"))
        converted.append((dst, rc.convert_notebook(os.path.join(L["uploads"], f), dst)))
for dst, swapped in converted:
    print(("converted  " if swapped else "copied (no Kaggle set-up cell found)  ") + dst)
print(f"{len(converted)} notebooks in {L['notebooks_colab']}")

## Step 7 · Check everything

In [ ]:
import json
import pandas as pd
if L["oodlab_code"] not in sys.path:
    sys.path.insert(0, L["oodlab_code"])
import oodlab
rows, bank, cache_dir = rc.verify_cache(L["cache"], "ViT-B/16")
print(pd.DataFrame(rows).to_string(index=False))
checks = {
    "oodlab code": os.path.isfile(os.path.join(L["oodlab_code"], "oodlab", "__init__.py")),
    "reneg code": os.path.isfile(os.path.join(L["reneg_code"], "reneg", "__init__.py")),
    "all 12 feature sets": all(r["status"] == "OK" for r in rows),
    "text bank: 69,554 words, 10,000 negatives": bool(bank) and bank.get("n_corpus") == 69554 and bank.get("n_selected") == 10000,
    "TANL results from notebook 02": os.path.isfile(os.path.join(L["results"], "02_reproduce_tanl", "tanl_runs.csv")),
    "C1 and G1 notebooks on Drive": all(os.path.isfile(os.path.join(L["notebooks"], n))
                                         for n in ("C1_colab_setup_check.ipynb", "G1_transport_gate.ipynb")),
}
print()
for k, v in checks.items():
    print(("PASS  " if v else "FAIL  ") + k)
print("\ntext bank:", json.dumps(bank) if bank else "missing")
with open(os.path.join(L["root"], "transfer_report.json"), "w") as f:
    json.dump({"oodlab": oodlab.__version__, "cache_dir": cache_dir, "sets": rows, "bank": bank, "checks": checks}, f, indent=1)
print("\nALL DONE -> open MyDrive/ReNeg/notebooks/C1_colab_setup_check.ipynb" if all(checks.values())
      else "\nSomething is missing -> see the FAIL lines and GUIDE_COLAB.md, troubleshooting")

**After C1 passes** you can delete `MyDrive/ReNeg/transfers/` to free Drive space; everything needed was copied
out of it. Run this notebook again later to bring over the ViT-L/14 cache (checklist step 3.1).

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")